# Agent State Machine (ASM) - Quick Start Guide

An **agent** is a wrapper around a finite state machine designed to accomplish a specific task and will be referred to as ASM.

A **StateMachineBuilder** is used to build the ASM from a manifest and a state diagram.

```mermaid
flowchart TD
    A["state_diagram"] -->|Input| B["StateMachineBuilder"]
    A2["state_manifest"] -->|Input| C["StateModel"]
    C -->|Build| B
    B --> D["fsm"]
```


---

## Example: Standard Agent

In this example, we will demonstrate how to create a simple **Agent** agent using a state diagram and state manifest.  
The state logic is implemented using **PureActionState** which allows custom actions to be attached to states.

### a) Define State Diagram

The following is a simple example of an **Agent** agent using 3 states:

* **INIT:** Collect initial input.
* **GENERATE:** The state where the LLM generates a response.
* **FINAL:** Return final output.

```mermaid
stateDiagram-v2
direction LR
INIT --> GENERATE: next / action
GENERATE --> FINAL: next / action
```


In [1]:
STATE_DIAGRAM = """
    INIT --> CHAT
    CHAT --> FINAL
    """

### b) Define State Manifest

The state manifest is a dictionary. 
Each state in the manifest corresponds to a state in the state diagram.


In [2]:
STATE_MANIFEST_V1 = {
    "INIT": {
        "input_data": {
            "llm_config": {"type": "getter", "dependency": "get_llm_config"}
            }
    },
    "CHAT": {
        "module_path": "gai.asm.states",
        "class_name": "PureActionState",
        "title": "CHAT",
        "action": "generate",
        "input_data": {
            "llm_config": {"type": "state_bag", "dependency": "llm_config"},
        },
        "output_data": ["streamer", "get_assistant_message"],
    },
    "FINAL": {"output_data": ["monologue"]},
}


### c) Create state action

In [3]:
from gai.llm.openai import AsyncOpenAI

async def generate_action(state):
    
    llm_config = state.input["llm_config"]
    client = AsyncOpenAI(llm_config)
    
    # Get message from input
    user_message = state.machine.user_message
    
    # Execute
    response = await client.chat.completions.create(
        model=llm_config["model"],
        messages=[{
            "role":"user",
            "content":user_message
            }],
        max_tokens=50,
        stream=True
    )
    
    assistant_message = ""
    async def streamer():
        nonlocal assistant_message
        async for chunk in response:
            if chunk:
                chunk = chunk.choices[0].delta.content
                if isinstance(chunk,str) and chunk:
                    assistant_message += chunk
                    yield chunk

    state.machine.state_bag["get_assistant_message"] = lambda: assistant_message
    state.machine.state_bag["streamer"] = streamer()

### c) Build State Machine

In [4]:
from gai.asm import AsyncStateMachine

with AsyncStateMachine.StateMachineBuilder(STATE_DIAGRAM) as builder:
    fsm = builder.build(
        STATE_MANIFEST_V1,
        get_llm_config=lambda state: {
            "client_type": "anthropic",
            "model": "claude-opus-4-20250514",
        },
        # get_llm_config=lambda state: {
        #     "client_type": "gai",
        #     "model": "ttt",
        #     "url": "http://gai-llm-svr:12031/gen/v1/chat/completions"
        # },
        agent_name="Sara",
        generate=generate_action,
    )
    


### d) Run State Machine (INIT->CHAT)

In [5]:
fsm.user_message = "Write a one sentence story"
await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    print(chunk,end='',flush=True)
print("\n\n")

The old lighthouse keeper realized, as the last boat disappeared beyond the horizon, that he had forgotten to tell anyone he was retiring.




### d) Continue (CHAT->FINAL)

In [6]:
await fsm.run_async()
print("State History:")
for state in fsm.state_history:
    print(f"State: {state['state']}")
    print(f"- input: {state['input']}")
    print(f"- output: {state['output']}")
    print("-" * 20)
print("Assistant Message:")
fsm.state_bag["get_assistant_message"]()

State History:
State: INIT
- input: {'user_message': 'Write a one sentence story', 'monologue': <gai.messages.monologue.Monologue object at 0x7241a8e08040>, 'step': 0, 'time': datetime.datetime(2025, 7, 14, 12, 10, 35, 908944), 'name': 'Sara', 'llm_config': {'client_type': 'anthropic', 'model': 'claude-opus-4-20250514'}}
- output: {'name': 'Sara', 'user_message': 'Write a one sentence story', 'monologue': <gai.messages.monologue.Monologue object at 0x7241a8e82380>, 'step': 1, 'time': datetime.datetime(2025, 7, 14, 12, 10, 35, 909006)}
--------------------
State: CHAT
- input: {'name': 'Sara', 'user_message': 'Write a one sentence story', 'monologue': <gai.messages.monologue.Monologue object at 0x7241a8e083a0>, 'step': 1, 'time': datetime.datetime(2025, 7, 14, 12, 10, 35, 909177), 'llm_config': {'client_type': 'anthropic', 'model': 'claude-opus-4-20250514'}}
- output: {'streamer': <async_generator object generate_action.<locals>.streamer at 0x7241a87e32c0>, 'get_assistant_message': <fun

'The old lighthouse keeper realized, as the last boat disappeared beyond the horizon, that he had forgotten to tell anyone he was retiring.'

### Complete Code Example (using ChatState and FileMonologue)

Here is another version of the example that replaces PureActionState with the built-in ChatState.
You can attach a FileMonologue to handle in-state memory.


In [7]:
import os
from gai.asm import AsyncStateMachine
from gai.messages import FileMonologue
from gai.lib.tests import make_local_tmp
here = make_local_tmp()
file_path = os.path.join(here,"monologue.json")
monologue = FileMonologue(file_path=file_path)
STATE_DIAGRAM = """
    INIT --> CHAT
    CHAT --> FINAL
    """
STATE_MANIFEST_V1 = {
    "INIT": {
        "input_data": {
            "llm_config": {"type": "getter", "dependency": "get_llm_config"}
            }
    },
    "CHAT": {
        "module_path": "gai.asm.states",
        "class_name": "ChatState",
        "title": "CHAT",
        "input_data": {
            "llm_config": {"type": "state_bag", "dependency": "llm_config"},
        },
        "output_data": ["streamer", "get_assistant_message"],
    },    
    "FINAL": {"output_data": ["monologue"]},
}
with AsyncStateMachine.StateMachineBuilder(STATE_DIAGRAM) as builder:
    fsm = builder.build(
        STATE_MANIFEST_V1,
        get_llm_config=lambda state: {
            "client_type": "anthropic",
            "model": "claude-opus-4-20250514",
        },
        agent_name="Sara",
        monologue=monologue
    )
fsm.user_message = "Write a one sentence story"
await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    print(chunk,end='',flush=True)
print("\n\n")


The old lighthouse keeper discovered the beacon he'd faithfully tended for forty years had been automated a decade ago, but he climbed the spiral stairs each night anyway, unwilling to abandon his post to the darkness.[{'citations': None, 'text': "The old lighthouse keeper discovered the beacon he'd faithfully tended for forty years had been automated a decade ago, but he climbed the spiral stairs each night anyway, unwilling to abandon his post to the darkness.", 'type': 'text'}]




## Check Monologue

In [8]:
with open(file_path,"r") as f:
    txt = f.read()
print(txt)

{
    "next_message_order": 2,
    "messages": [
        {
            "id": "ff4e2ba3-98a2-4263-b711-ab979c13df94",
            "header": {
                "sender": "User",
                "recipient": "Assistant",
                "timestamp": 1752495047.698685,
                "order": 0
            },
            "body": {
                "type": "monologue",
                "state_name": "CHAT",
                "step_no": 1,
                "content_type": "text",
                "role": "user",
                "content": "Write a one sentence story"
            }
        },
        {
            "id": "da4dab5f-8c53-4b9d-8b33-1a6f6db07f5a",
            "header": {
                "sender": "Assistant",
                "recipient": "User",
                "timestamp": 1752495053.643684,
                "order": 1
            },
            "body": {
                "type": "monologue",
                "state_name": "CHAT",
                "step_no": 1,
                "content_typ

---

## Example: Standard Agent (Part 2)

Same example but with an additional state to demonstrate context management by monologue messages.

```mermaid
stateDiagram-v2
direction LR
INIT --> CHAT
CHAT --> CONTINUE
CONTINUE --> FINAL
```


In [ ]:
import os
from gai.llm.openai import AsyncOpenAI
from gai.asm import AsyncStateMachine
from gai.messages import FileMonologue
from gai.lib.tests import make_local_tmp
here = make_local_tmp()
file_path = os.path.join(here,"monologue.json")
monologue = FileMonologue(file_path=file_path)

STATE_DIAGRAM = """
    INIT --> CHAT
    CHAT --> CONTINUE
    CONTINUE --> FINAL
    """
    
STATE_MANIFEST_V1 = {
    "INIT": {
        "input_data": {
            "llm_config": {"type": "getter", "dependency": "get_llm_config"},
        }
    },
    "CHAT": {
        "module_path": "gai.asm.states",
        "class_name": "ChatState",
        "title": "CHAT",
        "input_data": {
            "llm_config": {"type": "state_bag", "dependency": "llm_config"},
        },
        "output_data": ["streamer", "get_assistant_message"],
    },
    "CONTINUE": {
        "module_path": "gai.asm.states",
        "class_name": "PureActionState",
        "title": "CONTINUE",
        "action": "continue_action",
        "input_data": {
            "llm_config": {"type": "state_bag", "dependency": "llm_config"},
        },
        "output_data": ["streamer", "get_assistant_message"],
    },
    "FINAL": {"output_data": ["monologue"]},
}

async def continue_action(state):
    
    llm_config = state.input["llm_config"]
    client = AsyncOpenAI(llm_config)
    
    # Import data from state_bag
    state.machine.monologue.add_user_message(state=state,content="Please continue.")
    
    # Execute
    
    from gai.messages import message_helper
    messages = state.machine.monologue.list_messages()
    chat_messages = message_helper.convert_to_chat_messages(messages)
    chat_messages = message_helper.shrink_messages(chat_messages)
    
    response = await client.chat.completions.create(
        model=llm_config["model"],
        messages=chat_messages,
        max_tokens=50,
        stream=True
    )
    
    assistant_message = ""
    async def streamer():
        nonlocal assistant_message
        async for chunk in response:
            if chunk:
                chunk = chunk.choices[0].delta.content
                if isinstance(chunk,str) and chunk:
                    assistant_message += chunk
                    yield chunk
        state.machine.monologue.add_assistant_message(
            state=state, content=assistant_message
        )
        # Need to update the stale history due to delayed output
        state.machine.state_history[-1]["output"]["monologue"] = (
            state.machine.monologue.copy()
        )
    
    state.machine.state_bag["get_assistant_message"] = lambda: assistant_message
    state.machine.state_bag["streamer"] = streamer()
    
from gai.asm import AsyncStateMachine

with AsyncStateMachine.StateMachineBuilder(STATE_DIAGRAM) as builder:
    fsm = builder.build(
        STATE_MANIFEST_V1,
        get_llm_config=lambda state: {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
        },
        monologue=monologue,        
        agent_name="Sara",
        continue_action=continue_action,
    )


### a) Run State Machine (INIT->CHAT)

In [10]:
fsm.user_message = "Write a one sentence story"
await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    print(chunk,end='',flush=True)
print("\n\n")

print("Step 1:")
print("Before:")
for message in fsm.state_history[1]["input"]["monologue"].list_messages():
    print(
        f"{message.header.timestamp} {message.header.sender} > {message.body.content}"
    )

print("After:")
for message in fsm.state_history[1]["output"]["monologue"].list_messages():
    print(
        f"{message.header.timestamp} {message.header.sender} > {message.body.content}"
    )

The last star in the universe flickered out just as the gardener finished planting seeds in the cosmic void, whispering "let there be light" to the darkness.[{'citations': None, 'text': 'The last star in the universe flickered out just as the gardener finished planting seeds in the cosmic void, whispering "let there be light" to the darkness.', 'type': 'text'}]


Step 1:
Before:
1752388738.8597424 User > Write a one sentence story
1752388742.2931294 Assistant > [{'citations': None, 'text': 'The last star in the universe flickered out just as the gardener finished planting seeds in the cosmic void, whispering "let there be light" to the darkness.', 'type': 'text'}]
After:
1752388738.8597424 User > Write a one sentence story
1752388742.2931294 Assistant > [{'citations': None, 'text': 'The last star in the universe flickered out just as the gardener finished planting seeds in the cosmic void, whispering "let there be light" to the darkness.', 'type': 'text'}]


### c) Run State Machine (GENERATE->CONTINUE)

In [5]:
await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    print(chunk,end='',flush=True)
print("\n\n")

print("Step 2:")
print("Before:")
for message in fsm.state_history[2]["input"]["monologue"].list_messages():
    print(
        f"{message.header.timestamp} {message.header.sender} > {message.body.content}"
    )

print("After:")
for message in fsm.state_history[2]["output"]["monologue"].list_messages():
    print(
        f"{message.header.timestamp} {message.header.sender} > {message.body.content}"
    )

She had spent three years alone, tending to the atmospheric processors that finally scrubbed the toxic clouds from her home planet's sky, and now as the station's life support systems gave their final wheeze, Commander Sarah Chen pressed her palm against


Step 2:
Before:
1752385508.3851202 User > Write a one sentence story
1752385511.8883715 Assistant > [{'citations': None, 'text': "The last astronaut on the dying space station smiled as Earth's first sunrise in a thousand years painted the sky gold through her porthole.", 'type': 'text'}]
1752385517.1696405 User > Please continue.
1752385521.4583452 Assistant > She had spent three years alone, tending to the atmospheric processors that finally scrubbed the toxic clouds from her home planet's sky, and now as the station's life support systems gave their final wheeze, Commander Sarah Chen pressed her palm against
After:
1752385508.3851202 User > Write a one sentence story
1752385511.8883715 Assistant > [{'citations': None, 'text': "The

### e) END (GENERATE->FINAL)

In [6]:
await fsm.run_async()
print("State History:")
for state in fsm.state_history:
    print(f"State: {state['state']}")
    print(f"- input: {state['input']}")
    print(f"- output: {state['output']}")
    print("-" * 20)
print("Assistant Message:")
fsm.state_bag["get_assistant_message"]()

State History:
State: INIT
- input: {'user_message': 'Write a one sentence story', 'monologue': <gai.messages.monologue.FileMonologue object at 0x7d73815b9630>, 'step': 0, 'time': datetime.datetime(2025, 7, 13, 5, 45, 7, 375628), 'name': 'Sara', 'llm_config': {'client_type': 'anthropic', 'model': 'claude-sonnet-4-20250514'}}
- output: {'name': 'Sara', 'user_message': 'Write a one sentence story', 'monologue': <gai.messages.monologue.FileMonologue object at 0x7d73815b8be0>, 'step': 1, 'time': datetime.datetime(2025, 7, 13, 5, 45, 7, 375716)}
--------------------
State: CHAT
- input: {'name': 'Sara', 'user_message': 'Write a one sentence story', 'monologue': <gai.messages.monologue.FileMonologue object at 0x7d73815b9720>, 'step': 1, 'time': datetime.datetime(2025, 7, 13, 5, 45, 7, 376008), 'llm_config': {'client_type': 'anthropic', 'model': 'claude-sonnet-4-20250514'}}
- output: {'streamer': <async_generator object ChatState.run_async.<locals>.streamer at 0x7d7380bc56c0>, 'name': 'Sara',

"She had spent three years alone, tending to the atmospheric processors that finally scrubbed the toxic clouds from her home planet's sky, and now as the station's life support systems gave their final wheeze, Commander Sarah Chen pressed her palm against"